# Shiny Dashboard Integration (Production Overview)

This notebook presents the final stage of the project: integrating the prediction pipeline into an interactive dashboard using R Shiny.

The dashboard enables users to:

- Visualize bike-sharing demand across multiple cities
- Identify high-demand locations using an interactive map
- Drill down into detailed forecasts for specific cities

This notebook focuses on explaining the **core architecture and logic** of the dashboard.

> ⚠️ Note:
> The complete production implementation (including themes, UI styling, additional controls, and enhancements) is provided separately in the project repository as `ui.R` and `server.R`.
> 
> To maintain readability, only the essential logic is documented here.

## System Integration Overview

The Shiny dashboard integrates multiple components developed throughout the project:

### 🔷 Data Layer
- Weather data retrieved via OpenWeather API

### 🔷 Model Layer
- Regression model trained on historical data

### 🔷 Prediction Layer
- Implemented in `model_prediction.R`

### 🔷 Presentation Layer
- R Shiny dashboard using Leaflet

This modular design ensures separation of concerns and maintainability.

## User Interface (UI) — Conceptual Structure

The UI defines:

- Layout of the application
- Input controls (dropdown selection)
- Output containers (Leaflet map)

The production UI includes:

- Shiny themes (Bootswatch Yeti)
- Styled buttons and layout enhancements
- Additional formatting for improved user experience

Below is a simplified version highlighting the core structure.

In [ ]:
# NOTE: This is a simplified representation of the UI structure

library(shiny)
library(leaflet)

ui <- fluidPage(
  titlePanel("Bike-sharing demand prediction app"),
  
  sidebarLayout(
    
    mainPanel(
      leafletOutput("city_bike_map", height = 1000)
    ),
    
    sidebarPanel(
      selectInput(
        "city_dropdown",
        "Select City:",
        choices = c("All", "Seoul", "Suzhou", "London", "New York", "Paris"),
        selected = "All"
      )
    )
  )
)

## Server Logic — Core Workflow

The server performs the following operations:

1. Generates prediction data using API + model  
2. Aggregates data for overview visualization  
3. Applies conditional logic based on user input  
4. Renders Leaflet map dynamically  

The production server includes additional enhancements such as:

- Styled UI elements
- Extended plotting (Part B)
- Improved labeling and formatting

Below is a simplified version focusing on core logic.

In [ ]:
# NOTE: This is a simplified representation of the server logic

library(shiny)
library(leaflet)
library(dplyr)

source("model_prediction.R")

server <- function(input, output) {
  
  # Generate dataset
  city_weather_bike_df <- generate_city_weather_bike_data()
  
  # Aggregate data for overview map
  cities_max_bike <- city_weather_bike_df %>%
    group_by(CITY_ASCII, LAT, LNG) %>%
    summarise(
      BIKE_PREDICTION = max(BIKE_PREDICTION, na.rm = TRUE),
      BIKE_PREDICTION_LEVEL = BIKE_PREDICTION_LEVEL[which.max(BIKE_PREDICTION)],
      LABEL = LABEL[which.max(BIKE_PREDICTION)],
      DETAILED_LABEL = DETAILED_LABEL[which.max(BIKE_PREDICTION)],
      .groups = "drop"
    )
  
  # Render map
  output$city_bike_map <- renderLeaflet({
    
    if (input$city_dropdown == "All") {
      
      leaflet(data = cities_max_bike) %>%
        addTiles() %>%
        addCircleMarkers(
          lng = ~LNG,
          lat = ~LAT,
          popup = ~LABEL
        )
      
    } else {
      
      selected_city <- city_weather_bike_df %>%
        filter(CITY_ASCII == input$city_dropdown)
      
      leaflet(data = selected_city) %>%
        addTiles() %>%
        addMarkers(
          lng = ~LNG,
          lat = ~LAT,
          popup = ~DETAILED_LABEL
        )
    }
  })
}

## Full Application Code Availability

The complete Shiny application includes additional features not shown in this notebook, such as:

- Custom UI styling using Shiny themes (Bootswatch Yeti)
- Styled buttons and layout enhancements
- Extended visualizations (temperature trends, correlations)
- Enhanced popup formatting with detailed weather attributes

These elements are intentionally omitted here to maintain clarity and focus on the core system logic.

👉 The full production code is available in the repository:

- `ui.R`
- `server.R`
- `model_prediction.R`

Users are encouraged to refer to these files for the complete implementation.

## End-to-End Data Flow

The dashboard operates through the following pipeline:

1. API retrieves weather forecast data  
2. Model generates predictions  
3. Data is aggregated for visualization  
4. User input determines display mode  
5. Leaflet renders interactive output  

This creates a real-time, user-driven analytics system.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta </span>  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook provides an overview of the Shiny dashboard integration, focusing on core logic and system architecture.

The workflow follows <span style="color:blue">IBM Skills Network </span> instructional labs on dashboard development and deployment.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo </span>  
- <span style="color:blue">Jeff Grossman</span>  

---

## Project Context

This notebook represents the deployment stage of the end-to-end data science pipeline:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Model Development  
- Model Evaluation  
- Model Selection  
- API Integration & Prediction Pipeline  
- Dashboard Data Preparation  
- **Interactive Dashboard Deployment (R Shiny)**  

---

## Notes

The full production implementation of the dashboard is available in the repository and includes additional styling and advanced features beyond the core logic presented here.

---